In [28]:
#%pip install pandas numpy scikit-learn xgboost
#%pip install seaborn

In [29]:
 #SETUP & IMPORTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb

# Set plotting style for the research paper
plt.style.use('ggplot')
print("Libraries successfully imported!")

Libraries successfully imported!


In [30]:
# ==========================================
# CELL 2: DATA INGESTION & CLEANING (AUTO-SEARCH)

import os
import pandas as pd

# 1. Define column structure (No invisible spaces)
columns = [
    'Time_ms', 
    'Ankle_X', 'Ankle_Y', 'Ankle_Z', 
    'UpperLeg_X', 'UpperLeg_Y', 'UpperLeg_Z', 
    'Trunk_X', 'Trunk_Y', 'Trunk_Z', 
    'Label'
]

# 2. Automatically search for the file so we bypass path errors
base_dir = r'C:\Users\Red Moon\Coding\Harvard_SRI'
target_file = 'S01R01.txt'
file_path = None

print("Locating dataset...")
for root, dirs, files in os.walk(base_dir):
    if target_file in files:
        file_path = os.path.join(root, target_file)
        break

# Safety check just in case the file was accidentally deleted
if not file_path:
    raise FileNotFoundError(f"❌ ERROR: Could not find '{target_file}' anywhere inside {base_dir}.")

print(f"✅ File successfully located at: {file_path}")

# 3. Load the raw text file
df_raw = pd.read_csv(file_path, names=columns, delim_whitespace=True)
print(f"Original Raw Dataset Size: {df_raw.shape}")

# 4. Clean the data: Drop rows where Label == 0 (Not part of the experiment)
df_clean = df_raw[df_raw['Label'] != 0].copy()

# 5. Map the labels: 0 = Normal Walking, 1 = Freezing Episode
df_clean['Label'] = df_clean['Label'].map({1: 0, 2: 1})
print(f"Cleaned Dataset Size (Only Active Experiment): {df_clean.shape}")

# 6. Preview the dataframe
df_clean.head()

Locating dataset...
✅ File successfully located at: C:\Users\Red Moon\Coding\Harvard_SRI\dataset_fog_release\dataset\S01R01.txt
Original Raw Dataset Size: (151987, 11)
Cleaned Dataset Size (Only Active Experiment): (92802, 11)


,Time_ms,Ankle_X,Ankle_Y,Ankle_Z,UpperLeg_X,UpperLeg_Y,UpperLeg_Z,Trunk_X,Trunk_Y,Trunk_Z,Label
47999,750000,-30,990,326,-45,972,181,-38,1000,29,0
48000,750015,-30,1000,356,-18,981,212,-48,1028,29,0
48001,750031,-20,990,336,18,981,222,-38,1038,9,0
48002,750046,-20,1000,316,36,990,222,-19,1038,9,0
48003,750062,0,990,316,36,990,212,-29,1038,29,0


In [31]:

#  TEMPORAL FEATURE ENGINEERING


# 1. Define sliding window parameters
window_size = 128  # 128 rows = 2 seconds of data at 64Hz
step_size = 64     # 64 rows = 1 second overlap

# 2. Define the sensor columns we want to extract features from
sensor_columns = [
    'Ankle_X', 'Ankle_Y', 'Ankle_Z', 
    'UpperLeg_X', 'UpperLeg_Y', 'UpperLeg_Z', 
    'Trunk_X', 'Trunk_Y', 'Trunk_Z'
]

extracted_features = []
print("Engineering features from sliding windows... Please wait.")

# 3. Loop through the dataset in chunks
for i in range(0, len(df_clean) - window_size, step_size):
    window = df_clean.iloc[i : i + window_size]
    
    # If ANY row in this 2-second window is a freeze, mark the whole window as a freeze
    label = 1 if (window['Label'] == 1).any() else 0
        
    feature_dict = {'Label': label}
    
    # 4. Extract statistical features for ALL 9 axes
    for col in sensor_columns:
        sensor_data = window[col]
        feature_dict[f'{col}_Mean'] = np.mean(sensor_data)
        feature_dict[f'{col}_Std'] = np.std(sensor_data)
        feature_dict[f'{col}_Var'] = np.var(sensor_data)
        feature_dict[f'{col}_RMS'] = np.sqrt(np.mean(sensor_data**2))
        
    extracted_features.append(feature_dict)

# 5. Convert to a Machine-Learning-ready DataFrame
df_features = pd.DataFrame(extracted_features)
print(f"Feature Extraction Complete! New dataset shape: {df_features.shape}")

df_features.head()

Engineering features from sliding windows... Please wait.
Feature Extraction Complete! New dataset shape: (1449, 37)


,Label,Ankle_X_Mean,Ankle_X_Std,Ankle_X_Var,Ankle_X_RMS,Ankle_Y_Mean,Ankle_Y_Std,Ankle_Y_Var,Ankle_Y_RMS,Ankle_Z_Mean,...,Trunk_X_Var,Trunk_X_RMS,Trunk_Y_Mean,Trunk_Y_Std,Trunk_Y_Var,Trunk_Y_RMS,Trunk_Z_Mean,Trunk_Z_Std,Trunk_Z_Var,Trunk_Z_RMS
0,0,-49.937500,37.837182,1431.652344,62.653063,991.343750,14.973120,224.194336,991.456820,323.328125,...,962.466736,38.839312,1019.164062,17.373101,301.824646,1019.312126,10.148438,18.507638,342.532654,21.107426
1,0,-48.718750,31.716010,1005.905273,58.132795,989.625000,15.623000,244.078125,989.748311,329.304688,...,760.388428,36.525890,1017.062500,21.457716,460.433594,1017.288829,19.140625,21.883475,478.886475,29.073184
2,0,-50.414062,32.093741,1030.008240,59.762747,988.671875,15.311918,234.454834,988.790438,332.125000,...,788.069275,36.816797,1015.671875,21.094323,444.970459,1015.890904,19.742188,25.554551,653.035095,32.292245
3,0,-55.523438,31.496767,992.046326,63.834931,988.781250,15.897591,252.733398,988.909042,328.906250,...,724.354919,39.765425,1018.359375,17.080569,291.745850,1018.502608,1.140625,28.385938,805.761475,28.408845
4,0,-60.570312,28.451077,809.463806,66.919553,988.843750,16.925590,286.475586,988.988593,328.890625,...,530.616211,45.429375,1018.710938,19.256149,370.799255,1018.892916,-7.250000,21.382309,457.203125,22.577990


In [32]:

#  MODEL TRAINING & BASELINE COMPARISON


# 1. Separate features (X) and target label (y)
X = df_features.drop(columns=['Label'])
y = df_features['Label']

# 2. Split data into 70% training and 30% validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.30, random_state=42)
print(f"Training on {X_train.shape[0]} windows, Validating on {X_val.shape[0]} windows.\n")

# 3. Train and Evaluate Random Forest
print("--- MODEL 1: RANDOM FOREST ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_val)

print(f"Accuracy: {accuracy_score(y_val, rf_preds) * 100:.2f}%")
print(classification_report(y_val, rf_preds, target_names=['Normal (0)', 'Freeze (1)']))

# 4. Train and Evaluate XGBoost
print("\n--- MODEL 2: XGBOOST ---")
xgb_model = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', max_depth=6)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_val)

print(f"Accuracy: {accuracy_score(y_val, xgb_preds) * 100:.2f}%")
print(classification_report(y_val, xgb_preds, target_names=['Normal (0)', 'Freeze (1)']))

Training on 1014 windows, Validating on 435 windows.

--- MODEL 1: RANDOM FOREST ---
Accuracy: 94.02%
              precision    recall  f1-score   support

  Normal (0)       0.95      0.98      0.97       400
  Freeze (1)       0.71      0.43      0.54        35

    accuracy                           0.94       435
   macro avg       0.83      0.71      0.75       435
weighted avg       0.93      0.94      0.93       435


--- MODEL 2: XGBOOST ---
Accuracy: 92.64%
              precision    recall  f1-score   support

  Normal (0)       0.95      0.97      0.96       400
  Freeze (1)       0.56      0.40      0.47        35

    accuracy                           0.93       435
   macro avg       0.75      0.69      0.71       435
weighted avg       0.92      0.93      0.92       435

